In [3]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import requests
import io

In [4]:
#new match nasa function --> editing slightly
#script to matchup time series data & weather data from nasa
#from Izzy --> some slight adjustments to deal with change in time series data format 
def match_nasa(df, name):
    """
    match nasa weather data with dates and locations in the existing dataset
    """
    
    # load final matched dataframe
    df['satellite_date'] = pd.to_datetime(df['satellite_date'])
    df['sample_date'] = pd.to_datetime(df['sample_date'])

    weather_results = []
    
    # iterate through unique location/date combos
    unique_matchups = df[['latitude', 'longitude', 'sample_date', 'satellite_date']].drop_duplicates()

    print(f"fetching weather for {len(unique_matchups)} matchups...")

    # for each unique matchup, get weather data
    for _, row in tqdm(unique_matchups.iterrows(), total=len(unique_matchups)):

        lat, lon = row['latitude'], row['longitude']

        # convert dates for nasa power
        date_str = row['satellite_date'].strftime('%Y%m%d')
        
        # create window 3 days before so we can calculate accumulated precipitation
        start_date = (row['satellite_date'] - pd.Timedelta(days=3)).strftime('%Y%m%d')
        
        url = f"https://power.larc.nasa.gov/api/temporal/daily/point"
        params = {
            "parameters": "PRECTOTCORR,WS2M,ALLSKY_SFC_SW_DWN",
            "community": "RE",
            "longitude": lon,
            "latitude": lat,
            "start": start_date,
            "end": date_str,
            "format": "JSON"
        }

        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code == 200:

                # get data
                data = response.json()['properties']['parameter']
                
                # sum rain over the 3 days before
                precip_3d = sum(data['PRECTOTCORR'].values())

                # get wind and solar for the exact day
                wind = list(data['WS2M'].values())[-1]
                solar = list(data['ALLSKY_SFC_SW_DWN'].values())[-1]
                
                #add data to results
                weather_results.append({
                    'latitude': lat, 'longitude': lon, 'sample_date': row['sample_date'], 'satellite_date': row['satellite_date'],
                    'precip_3d_mm': precip_3d,
                    'wind_speed_ms': wind,
                    'solar_rad_wm2': solar
                })
            
            else:
                print(f"failed: Status {response.status_code} at {lat}, {lon}")

            time.sleep(0.2) 
            
        except Exception as e:
            print(lat, lon, row['satellite_date'])
            print("ERROR:", repr(e))
            continue

    # merge weather back into main dataframe
    weather_df = pd.DataFrame(weather_results)
    print(weather_df.columns)
    print(weather_df.shape)
    print(df.columns)
    print(df.shape)
    #merge after
    weather_df.to_csv(name, index=False)
    return weather_df

In [6]:
#accessing the four dfs
df_1week_8 = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/1week_forecast_noweather.csv')
df_2week_7 = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/2week_forecast_noweather.csv')
df_3week_7 = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/3week_forecast_noweather.csv')
df_1month_6 = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/1month_forecast_noweather.csv')

In [8]:
#good
print(df_1week_8.shape)
print(df_2week_7.shape)
print(df_3week_7.shape)
print(df_1month_6.shape)

(45656, 20)
(40985, 20)
(38717, 20)
(33156, 20)


In [10]:
#running matching scripts for each of these --> get match & unmatched
df_match = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/with_weather_50.csv')
def find_matches(df, df_match):
    join_cols = ['latitude', 'longitude', 'sample_date', 'satellite_date']
    
    #in datetime
    df['sample_date'] = pd.to_datetime(df['sample_date'])
    df_match['sample_date'] = pd.to_datetime(df_match['sample_date'])
    df['satellite_date'] = pd.to_datetime(df['satellite_date'])
    df_match['satellite_date'] = pd.to_datetime(df_match['satellite_date'])

    #rows with a match
    df_ww = df.merge(
        df_match,
        on=join_cols,
        how='inner'
    )
    #rows without a match
    df_nw = df.merge(
        df_match[join_cols],
        on=join_cols,
        how='left',
        indicator=True
    ).query('_merge == "left_only"').drop(columns=['_merge'])

    return df_ww, df_nw

In [11]:
#1 week
df_1week_ww_8, df_1week_nw_8 = find_matches(df_1week_8, df_match)
print(f"Shape of Matched Df: {df_1week_ww_8.shape}")
print(f"Missing rows: {np.abs(df_1week_ww_8.shape[0] - df_1week_8.shape[0])}")
print(f"Shape of Non-Matched Df: {df_1week_nw_8.shape}")
df_1week_ww_8.head(5)

Shape of Matched Df: (32458, 23)
Missing rows: 13198
Shape of Non-Matched Df: (13198, 20)


,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,...,red,red_edge,sample_date,satellite_date,.geo,mean_diff_days,std_diff_days,precip_3d_mm,wind_speed_ms,solar_rad_wm2
0,8665_20211215T185811_20211215T190030_T10SFF,0.102973,0.116336,0.041726,0.084986,5427.5,0.062534,36.772302,-121.788002,0.382538,...,0.076428,0.162559,2021-12-29,2021-12-15,"{""type"":""MultiPoint"",""coordinates"":[]}",5.714286,1.889822,94.83,1.97,2.6995
1,8664_20211215T185811_20211215T190030_T10SFF,0.110361,0.147759,0.038600,0.084986,2417.5,0.063480,36.796199,-121.781996,0.518339,...,0.062576,0.190656,2021-12-29,2021-12-15,"{""type"":""MultiPoint"",""coordinates"":[]}",5.714286,1.889822,94.83,1.97,2.6995
2,8664_20211210T185759_20211210T185754_T10SFF,0.116494,0.188641,0.052186,0.000064,2417.5,0.070992,36.796199,-121.781996,0.498272,...,0.073499,0.202982,2021-12-29,2021-12-10,"{""type"":""MultiPoint"",""coordinates"":[]}",5.714286,1.889822,6.39,1.32,3.1536
3,8665_20211210T185759_20211210T185754_T10SFF,0.093219,0.140802,0.045601,0.000064,5427.5,0.061202,36.772302,-121.788002,0.370419,...,0.068910,0.147982,2021-12-29,2021-12-10,"{""type"":""MultiPoint"",""coordinates"":[]}",5.714286,1.889822,6.39,1.32,3.1536
4,8389_20211207T184749_20211207T185252_T11SKU,0.411125,0.348128,0.496386,0.406460,2989.5,0.449381,34.406896,-119.517000,0.001044,...,0.407060,0.407676,2021-12-16,2021-12-07,"{""type"":""MultiPoint"",""coordinates"":[]}",5.000000,0.000000,0.29,0.60,2.2224


In [12]:
#2 week
df_2week_ww_7, df_2week_nw_7 = find_matches(df_2week_7, df_match)
print(f"Shape of Matched Df: {df_2week_ww_7.shape}")
print(f"Missing rows: {np.abs(df_2week_ww_7.shape[0] - df_2week_7.shape[0])}")
print(f"Shape of Non-Matched Df: {df_2week_nw_7.shape}")
df_2week_ww_7.head(5)

Shape of Matched Df: (26482, 23)
Missing rows: 14503
Shape of Non-Matched Df: (14503, 20)


,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,...,red,red_edge,sample_date,satellite_date,.geo,mean_diff_days,std_diff_days,precip_3d_mm,wind_speed_ms,solar_rad_wm2
0,8664_20211215T185811_20211215T190030_T10SFF,0.110361,0.147759,0.038600,0.084986,2417.5,0.063480,36.796199,-121.781996,0.518339,...,0.062576,0.190656,2021-12-29,2021-12-15,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,94.83,1.97,2.6995
1,8665_20211215T185811_20211215T190030_T10SFF,0.102973,0.116336,0.041726,0.084986,5427.5,0.062534,36.772302,-121.788002,0.382538,...,0.076428,0.162559,2021-12-29,2021-12-15,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,94.83,1.97,2.6995
2,8664_20211210T185759_20211210T185754_T10SFF,0.116494,0.188641,0.052186,0.000064,2417.5,0.070992,36.796199,-121.781996,0.498272,...,0.073499,0.202982,2021-12-29,2021-12-10,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,6.39,1.32,3.1536
3,8665_20211210T185759_20211210T185754_T10SFF,0.093219,0.140802,0.045601,0.000064,5427.5,0.061202,36.772302,-121.788002,0.370419,...,0.068910,0.147982,2021-12-29,2021-12-10,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,6.39,1.32,3.1536
4,8664_20211205T185751_20211205T185938_T10SFF,0.141857,0.206581,0.076861,0.194565,2417.5,0.097632,36.796199,-121.781996,0.420105,...,0.097001,0.220645,2021-12-29,2021-12-05,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,0.57,1.04,2.9026


In [13]:
#3 week
df_3week_ww_7, df_3week_nw_7 = find_matches(df_3week_7, df_match)
print(f"Shape of Matched Df: {df_3week_ww_7.shape}")
print(f"Missing rows: {np.abs(df_3week_ww_7.shape[0] - df_3week_7.shape[0])}")
print(f"Shape of Non-Matched Df: {df_3week_nw_7.shape}")
df_3week_ww_7.head(5)

Shape of Matched Df: (20587, 23)
Missing rows: 18130
Shape of Non-Matched Df: (18130, 20)


,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,...,red,red_edge,sample_date,satellite_date,.geo,mean_diff_days,std_diff_days,precip_3d_mm,wind_speed_ms,solar_rad_wm2
0,8664_20211205T185751_20211205T185938_T10SFF,0.141857,0.206581,0.076861,0.194565,2417.5,0.097632,36.796199,-121.781996,0.420105,...,0.097001,0.220645,2021-12-29,2021-12-05,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,0.57,1.04,2.9026
1,8665_20211205T185751_20211205T185938_T10SFF,0.106140,0.157354,0.060654,0.194565,5427.5,0.076743,36.772302,-121.788002,0.339411,...,0.083286,0.165783,2021-12-29,2021-12-05,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,0.57,1.04,2.9026
2,8665_20211130T185729_20211130T185732_T10SFF,0.095562,0.159001,0.047278,0.000006,5427.5,0.065381,36.772302,-121.788002,0.338575,...,0.073864,0.147435,2021-12-29,2021-11-30,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,0.00,0.93,3.2820
3,8664_20211130T185729_20211130T185732_T10SFF,0.156062,0.252560,0.086731,0.000006,2417.5,0.110747,36.796199,-121.781996,0.405099,...,0.117367,0.233665,2021-12-29,2021-11-30,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,0.00,0.93,3.2820
4,8665_20211125T185721_20211125T185907_T10SFF,0.094130,0.148896,0.036028,0.050810,5427.5,0.057713,36.772302,-121.788002,0.376554,...,0.066268,0.145182,2021-12-29,2021-11-25,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,0.02,1.05,3.4049


In [14]:
#4 week
df_1month_ww_6, df_1month_nw_6 = find_matches(df_1month_6, df_match)
print(f"Shape of Matched Df: {df_1month_ww_6.shape}")
print(f"Missing rows: {np.abs(df_1month_ww_6.shape[0] - df_1month_6.shape[0])}")
print(f"Shape of Non-Matched Df: {df_1month_nw_6.shape}")
df_1month_ww_6.head(5)

Shape of Matched Df: (13985, 23)
Missing rows: 19171
Shape of Non-Matched Df: (19171, 20)


,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,...,red,red_edge,sample_date,satellite_date,.geo,mean_diff_days,std_diff_days,precip_3d_mm,wind_speed_ms,solar_rad_wm2
0,8665_20211125T185721_20211125T185907_T10SFF,0.094130,0.148896,0.036028,0.050810,5427.5,0.057713,36.772302,-121.788002,0.376554,...,0.066268,0.145182,2021-12-29,2021-11-25,"{""type"":""MultiPoint"",""coordinates"":[]}",6.0,2.236068,0.02,1.05,3.4049
1,8664_20211125T185721_20211125T185907_T10SFF,0.134578,0.219440,0.062453,0.050810,2417.5,0.090117,36.796199,-121.781996,0.454228,...,0.094136,0.213985,2021-12-29,2021-11-25,"{""type"":""MultiPoint"",""coordinates"":[]}",6.0,2.236068,0.02,1.05,3.4049
2,8664_20211120T185659_20211120T185659_T10SEF,0.154327,0.252859,0.082347,0.077712,2417.5,0.106273,36.796199,-121.781996,0.400240,...,0.114535,0.225297,2021-12-29,2021-11-20,"{""type"":""MultiPoint"",""coordinates"":[]}",6.0,2.236068,0.60,1.12,3.4092
3,8665_20211120T185659_20211120T185659_T10SEF,0.099594,0.162399,0.046088,0.077712,5427.5,0.065280,36.772302,-121.788002,0.330881,...,0.072793,0.140715,2021-12-29,2021-11-20,"{""type"":""MultiPoint"",""coordinates"":[]}",6.0,2.236068,0.60,1.12,3.4092
4,8394_20211117T184649_20211117T185218_T10SGD,0.502881,0.613433,0.435473,0.263595,3792.0,0.451330,34.957600,-120.630995,0.124779,...,0.465398,0.597224,2021-12-17,2021-11-17,"{""type"":""MultiPoint"",""coordinates"":[]}",5.0,0.000000,0.03,1.80,3.0446


Making API calls to fill in these dataframes

In [17]:
#1 week --> it works
print(df_1week_nw_8.shape)
weather_week1 = match_nasa(df_1week_nw_8, "week1_weather_restofmatches.csv")

(13198, 20)
fetching weather for 13198 matchups...


100%|███████████████████████████████████| 13198/13198 [3:08:03<00:00,  1.17it/s]

Index(['latitude', 'longitude', 'sample_date', 'satellite_date',
       'precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'],
      dtype='object')
(13198, 7)
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days', 'std_diff_days'],
      dtype='object')
(13198, 20)


In [19]:
#rematch on the next attempt (hopefully shorter) ---> and so on
print(df_1week_nw_8.shape)
print(weather_week1.shape)

(13198, 20)
(13198, 7)


In [22]:
join_cols = ['latitude', 'longitude', 'sample_date', 'satellite_date']
df_1week_pt2 = df_1week_nw_8.merge(weather_week1, on = join_cols, how = 'left')
df_1week_pt2.head(5)
print(df_1week_pt2.shape)

(13198, 23)


In [24]:
print(df_1week_ww_8.shape)
print(df_1week_ww_8.columns)
print(df_1week_pt2.columns)

(32458, 23)
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days', 'std_diff_days',
       'precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'],
      dtype='object')
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days', 'std_diff_days',
       'precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'],
      dtype='object')


In [25]:
df_1week_pt2 = df_1week_pt2[df_1week_ww_8.columns]
week1_ww_8 = pd.concat([df_1week_ww_8, df_1week_pt2], ignore_index = True)
print(week1_ww_8.shape)
#week1_ww_8 contains the final w/ weather dataset

(45656, 23)


Find Crossover for week 2 --> make weather dataset

In [27]:
print(df_match.shape)
print(weather_week1.shape)
#concat these two
new_match = pd.concat([df_match, weather_week1], ignore_index = True)
print(new_match.shape)

(41040, 7)
(13198, 7)
(54238, 7)


In [28]:
#find match 
df_2week_ww_7, df_2week_nw_7 = find_matches(df_2week_7, new_match)
print(f"Shape of Matched Df: {df_2week_ww_7.shape}")
print(f"Missing rows: {np.abs(df_2week_ww_7.shape[0] - df_2week_7.shape[0])}")
print(f"Shape of Non-Matched Df: {df_2week_nw_7.shape}")
df_2week_ww_7.head(5)

Shape of Matched Df: (37646, 23)
Missing rows: 3339
Shape of Non-Matched Df: (3339, 20)


,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,...,red,red_edge,sample_date,satellite_date,.geo,mean_diff_days,std_diff_days,precip_3d_mm,wind_speed_ms,solar_rad_wm2
0,8664_20211215T185811_20211215T190030_T10SFF,0.110361,0.147759,0.038600,0.084986,2417.5,0.063480,36.796199,-121.781996,0.518339,...,0.062576,0.190656,2021-12-29,2021-12-15,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,94.83,1.97,2.6995
1,8665_20211215T185811_20211215T190030_T10SFF,0.102973,0.116336,0.041726,0.084986,5427.5,0.062534,36.772302,-121.788002,0.382538,...,0.076428,0.162559,2021-12-29,2021-12-15,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,94.83,1.97,2.6995
2,8664_20211210T185759_20211210T185754_T10SFF,0.116494,0.188641,0.052186,0.000064,2417.5,0.070992,36.796199,-121.781996,0.498272,...,0.073499,0.202982,2021-12-29,2021-12-10,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,6.39,1.32,3.1536
3,8665_20211210T185759_20211210T185754_T10SFF,0.093219,0.140802,0.045601,0.000064,5427.5,0.061202,36.772302,-121.788002,0.370419,...,0.068910,0.147982,2021-12-29,2021-12-10,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,6.39,1.32,3.1536
4,8664_20211205T185751_20211205T185938_T10SFF,0.141857,0.206581,0.076861,0.194565,2417.5,0.097632,36.796199,-121.781996,0.420105,...,0.097001,0.220645,2021-12-29,2021-12-05,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,0.57,1.04,2.9026


In [29]:
#make call to Nasa for the code
print(df_2week_nw_7.shape)
weather_week2 = match_nasa(df_2week_nw_7, "week2_weather_restofmatches.csv")

(3339, 20)
fetching weather for 3339 matchups...


  6%|██▌                                     | 214/3339 [03:07<45:08,  1.15it/s]

failed: Status 429 at 38.494202059810675, -121.5559999144485


100%|███████████████████████████████████████| 3339/3339 [48:23<00:00,  1.15it/s]

Index(['latitude', 'longitude', 'sample_date', 'satellite_date',
       'precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'],
      dtype='object')
(3338, 7)
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days', 'std_diff_days'],
      dtype='object')
(3339, 20)


In [44]:

join_cols = ['latitude', 'longitude', 'sample_date', 'satellite_date']
df_2week_pt2 = df_2week_nw_7.merge(weather_week2, on = join_cols, how = 'left', indicator = True)
#finding the one missing one
missing = df_2week_pt2[df_2week_pt2['_merge'] == 'left_only']
df_2week_pt2 = df_2week_pt2.drop(columns=['_merge'])
missing = missing.drop(columns=['_merge'])
print(df_2week_pt2.shape)
print(missing.shape)

(3339, 23)
(1, 23)


In [32]:
#make call on the missing column
missing_weather = match_nasa(missing, "temp")

fetching weather for 1 matchups...


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.08it/s]

Index(['latitude', 'longitude', 'sample_date', 'satellite_date',
       'precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'],
      dtype='object')
(1, 7)
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days', 'std_diff_days',
       'precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'],
      dtype='object')
(1, 23)


In [45]:
missing = missing.drop(columns = ['precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'])
print(missing.shape)

(1, 20)


In [46]:
missing = missing.merge(missing_weather, on = join_cols, how = 'left')
print(missing.shape)

(1, 23)


In [47]:
print(missing)
print(df_2week_pt2.shape)
join_cols = ['latitude', 'longitude', 'sample_date', 'satellite_date']
df_2week_pt2.set_index(join_cols, inplace=True)
missing.set_index(join_cols, inplace=True)
df_2week_pt2.update(missing)
df_2week_pt2.reset_index(inplace=True)
print(df_2week_pt2.shape)
print(df_2week_pt2.isna().sum())

                                 system:index       NIR      SWIR      blue  \
0  591_20210723T184919_20210723T185617_T10SFH  0.094665  0.142388  0.056082   

   cloud_score  cyanobacteria_abundance     green   latitude  longitude  \
0     0.000005                  4447.75  0.075193  38.494202   -121.556   

       ndci  ...       red  red_edge  sample_date  satellite_date  \
0  0.183312  ...  0.075762  0.140283   2021-09-13      2021-07-23   

                                     .geo mean_diff_days std_diff_days  \
0  {"type":"MultiPoint","coordinates":[]}       5.833333      2.041241   

  precip_3d_mm  wind_speed_ms  solar_rad_wm2  
0          0.0           2.45         8.5174  

[1 rows x 23 columns]
(3339, 23)
(3339, 23)
latitude                   0
longitude                  0
sample_date                0
satellite_date             0
system:index               0
NIR                        0
SWIR                       0
blue                       0
cloud_score                0
cy

In [48]:
print(df_2week_ww_7.shape)
print(df_2week_pt2.shape)

(37646, 23)
(3339, 23)


In [49]:
#now merge back to original dataframe and return result as csv
df_2week_pt2 = df_2week_pt2[df_2week_ww_7.columns]
week2_ww_7 = pd.concat([df_2week_ww_7, df_2week_pt2], ignore_index = True)
print(week2_ww_7.shape)
#week1_ww_8 contains the final w/ weather dataset

(40985, 23)


In [53]:
#return as csv for week1 and week2
#week1_ww_8.to_csv('week1_ww_8.csv', index = False)
#week2_ww_7.to_csv('week2_ww_7.csv', index = False)

In [55]:
print(weather_week2.isna().sum())
print(weather_week2.shape)

latitude          0
longitude         0
sample_date       0
satellite_date    0
precip_3d_mm      0
wind_speed_ms     0
solar_rad_wm2     0
dtype: int64
(3338, 7)


In [57]:
#update master matches
new_match = pd.concat([df_match, weather_week1], ignore_index = True)
master_matches = pd.concat([new_match, weather_week2], ignore_index = True).drop_duplicates()
print(master_matches.shape)

(57576, 7)


In [6]:
#newest matches
current_matches = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/time_series_model/all_matches_from_ts.csv')
print(current_matches.shape)

(61259, 7)


In [63]:
print(df_3week_7.shape)

(38717, 20)


In [64]:
#now find matches for week 3 and make necessary calls
df_3week_ww_7, df_3week_nw_7 = find_matches(df_3week_7, current_matches)
print(f"Shape of Matched Df: {df_3week_ww_7.shape}")
print(f"Missing rows: {np.abs(df_3week_ww_7.shape[0] - df_3week_7.shape[0])}")
print(f"Shape of Non-Matched Df: {df_3week_nw_7.shape}")
df_3week_ww_7.head(5)

Shape of Matched Df: (33426, 23)
Missing rows: 5291
Shape of Non-Matched Df: (5291, 20)


,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,...,red,red_edge,sample_date,satellite_date,.geo,mean_diff_days,std_diff_days,precip_3d_mm,wind_speed_ms,solar_rad_wm2
0,8664_20211205T185751_20211205T185938_T10SFF,0.141857,0.206581,0.076861,0.194565,2417.5,0.097632,36.796199,-121.781996,0.420105,...,0.097001,0.220645,2021-12-29,2021-12-05,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,0.57,1.04,2.9026
1,8665_20211205T185751_20211205T185938_T10SFF,0.106140,0.157354,0.060654,0.194565,5427.5,0.076743,36.772302,-121.788002,0.339411,...,0.083286,0.165783,2021-12-29,2021-12-05,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,0.57,1.04,2.9026
2,8665_20211130T185729_20211130T185732_T10SFF,0.095562,0.159001,0.047278,0.000006,5427.5,0.065381,36.772302,-121.788002,0.338575,...,0.073864,0.147435,2021-12-29,2021-11-30,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,0.00,0.93,3.2820
3,8664_20211130T185729_20211130T185732_T10SFF,0.156062,0.252560,0.086731,0.000006,2417.5,0.110747,36.796199,-121.781996,0.405099,...,0.117367,0.233665,2021-12-29,2021-11-30,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,0.00,0.93,3.2820
4,8665_20211125T185721_20211125T185907_T10SFF,0.094130,0.148896,0.036028,0.050810,5427.5,0.057713,36.772302,-121.788002,0.376554,...,0.066268,0.145182,2021-12-29,2021-11-25,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,0.02,1.05,3.4049


In [66]:
#TODO --> make call for 5291 missing points
print(df_3week_nw_7.shape)
weather_week3 = match_nasa(df_3week_nw_7, "week3_weather_restofmatches.csv")

(5291, 20)
fetching weather for 5291 matchups...


 53%|███████████████████▋                 | 2819/5291 [38:11<2:27:22,  3.58s/it]

40.41890168391374 -122.21399824319816 2019-07-17 00:00:00
ERROR: ReadTimeout(ReadTimeoutError("HTTPSConnectionPool(host='power.larc.nasa.gov', port=443): Read timed out. (read timeout=10)"))


100%|█████████████████████████████████████| 5291/5291 [1:11:41<00:00,  1.23it/s]

Index(['latitude', 'longitude', 'sample_date', 'satellite_date',
       'precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'],
      dtype='object')
(5290, 7)
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days', 'std_diff_days'],
      dtype='object')
(5291, 20)


In [7]:
current_matches = pd.concat([current_matches, weather_week3], ignore_index = True).drop_duplicates() 
print(current_matches.shape)

NameError: name 'weather_week3' is not defined

In [78]:
#error on one --> fix
join_cols = ['latitude', 'longitude', 'sample_date', 'satellite_date']
df_3week_pt2 = df_3week_nw_7.merge(weather_week3, on = join_cols, how = 'left', indicator = True)
#finding the one missing one
missing = df_3week_pt2[df_3week_pt2['_merge'] == 'left_only']
df_3week_pt2 = df_3week_pt2.drop(columns=['_merge'])
missing = missing.drop(columns=['_merge'])
print(df_3week_pt2.shape)
print(missing.shape)

(5291, 23)
(1, 23)


In [79]:
print(missing)

                                     system:index      NIR      SWIR  \
2818  4599_20190717T185929_20190717T191004_T10TEK  0.08168  0.173838   

          blue  cloud_score  cyanobacteria_abundance     green   latitude  \
2818  0.023663     0.144384                   6257.5  0.044224  40.418902   

       longitude      ndci  ...       red  red_edge  sample_date  \
2818 -122.213998  0.856372  ...  0.027086  0.357854   2019-08-21   

      satellite_date                                    .geo mean_diff_days  \
2818      2019-07-17  {"type":"MultiPoint","coordinates":[]}            2.5   

     std_diff_days precip_3d_mm  wind_speed_ms  solar_rad_wm2  
2818      0.547723          NaN            NaN            NaN  

[1 rows x 23 columns]


In [80]:
missing_weather = match_nasa(missing, "temp")

fetching weather for 1 matchups...


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.18it/s]

Index(['latitude', 'longitude', 'sample_date', 'satellite_date',
       'precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'],
      dtype='object')
(1, 7)
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days', 'std_diff_days',
       'precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'],
      dtype='object')
(1, 23)


In [81]:
print(missing_weather)

    latitude   longitude sample_date satellite_date  precip_3d_mm  \
0  40.418902 -122.213998  2019-08-21     2019-07-17           0.0   

   wind_speed_ms  solar_rad_wm2  
0            0.6         8.3688  


In [82]:
join_cols = ['latitude', 'longitude', 'sample_date', 'satellite_date']
df_3week_pt2.set_index(join_cols, inplace=True)
missing_weather.set_index(join_cols, inplace=True)
df_3week_pt2.update(missing_weather)
df_3week_pt2.reset_index(inplace=True)
print(df_3week_pt2.shape)
print(df_3week_pt2.isna().sum())

(5291, 23)
latitude                   0
longitude                  0
sample_date                0
satellite_date             0
system:index               0
NIR                        0
SWIR                       0
blue                       0
cloud_score                0
cyanobacteria_abundance    0
green                      0
ndci                       0
ndti                       0
ndvi                       0
ndwi                       0
red                        0
red_edge                   0
.geo                       0
mean_diff_days             0
std_diff_days              0
precip_3d_mm               0
wind_speed_ms              0
solar_rad_wm2              0
dtype: int64


In [84]:
print(df_3week_pt2.shape)
print(df_3week_ww_7.shape)

(5291, 23)
(33426, 23)


In [85]:
#now merge back to original dataframe and return result as csv
df_3week_pt2 = df_3week_pt2[df_3week_ww_7.columns]
week3_ww_7 = pd.concat([df_3week_ww_7, df_3week_pt2], ignore_index = True)
print(week3_ww_7.shape)

(38717, 23)


In [86]:
#add csv
#week3_ww_7.to_csv('week3_ww_7.csv', index = False)

#### Matches for 1 month dataset

In [5]:
df_1month_6 = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/1month_forecast_noweather.csv')
print(df_1month_6.shape)

(33156, 20)


In [9]:
#notebook got messed up --> recalling current_matches here
current_matches = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/time_series_model/all_matches_from_ts.csv')
recent_matches = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/week3_weather_restofmatches.csv')
print(current_matches.shape)
print(recent_matches.shape)
current_matches = pd.concat([current_matches, recent_matches], ignore_index = True).drop_duplicates()
print(current_matches.shape)

(61259, 7)
(5290, 7)
(66549, 7)


In [10]:
#running matching scripts for each of these --> get match & unmatched
def find_matches(df, df_match):
    join_cols = ['latitude', 'longitude', 'sample_date', 'satellite_date']
    
    #in datetime
    df['sample_date'] = pd.to_datetime(df['sample_date'])
    df_match['sample_date'] = pd.to_datetime(df_match['sample_date'])
    df['satellite_date'] = pd.to_datetime(df['satellite_date'])
    df_match['satellite_date'] = pd.to_datetime(df_match['satellite_date'])

    #rows with a match
    df_ww = df.merge(
        df_match,
        on=join_cols,
        how='inner'
    )
    #rows without a match
    df_nw = df.merge(
        df_match[join_cols],
        on=join_cols,
        how='left',
        indicator=True
    ).query('_merge == "left_only"').drop(columns=['_merge'])

    return df_ww, df_nw

In [11]:
#now find matches for week 3 and make necessary calls
df_1month_ww_6, df_1month_nw_6 = find_matches(df_1month_6, current_matches)
print(f"Shape of Matched Df: {df_1month_ww_6.shape}")
print(f"Missing rows: {np.abs(df_1month_ww_6.shape[0] - df_1month_6.shape[0])}")
print(f"Shape of Non-Matched Df: {df_1month_nw_6.shape}")
df_1month_ww_6.head(5)

Shape of Matched Df: (29120, 23)
Missing rows: 4036
Shape of Non-Matched Df: (4036, 20)


,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,...,red,red_edge,sample_date,satellite_date,.geo,mean_diff_days,std_diff_days,precip_3d_mm,wind_speed_ms,solar_rad_wm2
0,8665_20211125T185721_20211125T185907_T10SFF,0.094130,0.148896,0.036028,0.050810,5427.5,0.057713,36.772302,-121.788002,0.376554,...,0.066268,0.145182,2021-12-29,2021-11-25,"{""type"":""MultiPoint"",""coordinates"":[]}",6.0,2.236068,0.02,1.05,3.4049
1,8664_20211125T185721_20211125T185907_T10SFF,0.134578,0.219440,0.062453,0.050810,2417.5,0.090117,36.796199,-121.781996,0.454228,...,0.094136,0.213985,2021-12-29,2021-11-25,"{""type"":""MultiPoint"",""coordinates"":[]}",6.0,2.236068,0.02,1.05,3.4049
2,8664_20211120T185659_20211120T185659_T10SEF,0.154327,0.252859,0.082347,0.077712,2417.5,0.106273,36.796199,-121.781996,0.400240,...,0.114535,0.225297,2021-12-29,2021-11-20,"{""type"":""MultiPoint"",""coordinates"":[]}",6.0,2.236068,0.60,1.12,3.4092
3,8665_20211120T185659_20211120T185659_T10SEF,0.099594,0.162399,0.046088,0.077712,5427.5,0.065280,36.772302,-121.788002,0.330881,...,0.072793,0.140715,2021-12-29,2021-11-20,"{""type"":""MultiPoint"",""coordinates"":[]}",6.0,2.236068,0.60,1.12,3.4092
4,8394_20211117T184649_20211117T185218_T10SGD,0.502881,0.613433,0.435473,0.263595,3792.0,0.451330,34.957600,-120.630995,0.124779,...,0.465398,0.597224,2021-12-17,2021-11-17,"{""type"":""MultiPoint"",""coordinates"":[]}",5.0,0.000000,0.03,1.80,3.0446


In [12]:
#TODO --> make call for 4036 missing points
print(df_1month_nw_6.shape)
weather_month1 = match_nasa(df_1month_nw_6, "month1_weather_restofmatches.csv")

(4036, 20)
fetching weather for 4036 matchups...


100%|███████████████████████████████████████| 4036/4036 [54:13<00:00,  1.24it/s]

Index(['latitude', 'longitude', 'sample_date', 'satellite_date',
       'precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'],
      dtype='object')
(4036, 7)
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days', 'std_diff_days'],
      dtype='object')
(4036, 20)


In [13]:
join_cols = ['latitude', 'longitude', 'sample_date', 'satellite_date']
df_1month_pt2 = df_1month_nw_6.merge(weather_month1, on = join_cols, how = 'left')
print(df_1month_pt2.shape)

(4036, 23)


In [14]:
#now merge back to original dataframe and return result as csv
df_1month_pt2 = df_1month_pt2[df_1month_ww_6.columns]
month1_ww_6 = pd.concat([df_1month_ww_6, df_1month_pt2], ignore_index = True)
print(month1_ww_6.shape)

(33156, 23)


In [15]:
print(month1_ww_6.isna().sum().sum())

0


In [16]:
#export to csv
#month1_ww_6.to_csv('month1_ww_6.csv', index = False)